# 03 - SILVER: Limpieza y normalización / Cleaning and normalisation

## SPA

**Propósito**
Tomar la tabla bronze tal y como quedó en la ingesta y devolver el mismo dato corregido: tipos correctos, nombres de columna normalizados y nada más.

**Nota**

Esta capa es deliberadamente corta. El origen llegaba sin nulos, sin duplicados y con una sola columna mal tipada, de modo que había poco que limpiar. El trabajo de silver es proporcional a lo sucio que venga el dato de entrada, y aquí venía bien.

**Entrada**
`telco_churn.bronze.customers_raw` (7043 filas x 21 columnas, todas las columnas tal y como venían del CSV)

**Salida**
`telco_churn.silver.customers_clean` (7043 filas x 21 columnas, tipada y con nombres normalizados)

**Decisiones de diseño**

La regla que separa esta capa de la siguiente es sencilla: aquí se modifican columnas que ya existen, pero no se crea ninguna nueva. Todo lo que sea construir variables derivadas, contar servicios o calcular ratios pertenece a gold, porque son transformaciones pensadas para un consumidor concreto y no para cualquiera.

`TotalCharges` pasa a numérica. Los once valores vacíos se convierten en cero, porque corresponden a clientes con cero meses de antigüedad a los que todavía no se ha facturado nada. Es una regla de negocio determinista y no una imputación estadística, así que puede aplicarse antes de partir los datos sin riesgo de fuga.

La conversión trata únicamente los espacios en blanco y deja que cualquier otro valor no convertible provoque un error. Se descartó usar `errors="coerce"` precisamente para eso: si algún día el origen empieza a mandar valores en otro formato, el notebook debe pararse en vez de convertirlos en ceros.

Los nombres de columna pasan a `snake_case` en minúsculas. `Churn` se convierte a 0 y 1 por ser la variable objetivo, donde esa codificación es convención universal. El resto de variables categóricas se quedan como texto y se codifican dentro del pipeline del modelo, para que el objeto que se despliegue sepa transformar datos crudos por sí mismo.

---

## ENG

**Purpose**
Take the bronze table as the ingestion left it and return the same data corrected: proper types, normalised column names and nothing else.

**Note**

This layer is deliberately short. The source arrived with no nulls, no duplicates and a single badly typed column, so there was little to clean. The work done in silver is proportional to how dirty the incoming data is, and here it came in good shape.

**Input**
`telco_churn.bronze.customers_raw` (7,043 rows x 21 columns, every column exactly as it came from the CSV)

**Output**
`telco_churn.silver.customers_clean` (7,043 rows x 21 columns, typed and with normalised names)

**Design decisions**

The rule separating this layer from the next one is simple: existing columns may be modified here, but no new ones are created. Anything that builds derived features, counts services or computes ratios belongs in gold, since those are transformations aimed at one specific consumer rather than at everyone.

`TotalCharges` becomes numeric. The eleven blank values are converted to zero, because they belong to customers with zero months of tenure who have not been billed anything yet. This is a deterministic business rule rather than a statistical imputation, so it can be applied before splitting the data without any risk of leakage.

The conversion handles blank spaces only and lets any other unconvertible value raise an error. Using `errors="coerce"` was deliberately ruled out for that reason: if the source ever starts sending values in a different format, the notebook should stop rather than turning them into zeros.

Column names are converted to lowercase `snake_case`. `Churn` is mapped to 0 and 1 as the target variable, where that encoding is universal convention. The remaining categorical variables are left as text and encoded inside the model pipeline, so that the deployed object knows how to transform raw data on its own.

In [0]:
import pandas as pd
import numpy as np
import re

from churn.config import BRONZE_TABLE, SILVER_TABLE

In [0]:
pdf = spark.table(BRONZE_TABLE).toPandas()

In [0]:
def dataframe_asserts(pdf: pd.DataFrame):
  assert isinstance(pdf, pd.DataFrame), "El dataframe debe ser de tipo pandas"
  assert pdf.shape == (7043, 21), f"Forma inesperada: {pdf.shape}"
  assert not pdf["customerID"].duplicated().any(), "Se encontraron filas duplicadas"
  assert pdf["TotalCharges"].dtype == "object", "La columna TotalCharges debe ser de tipo object"
  assert pdf["Churn"].value_counts().shape == (2,), "La columna Churn debe tener dos valores únicos"


dataframe_asserts(pdf)


## 1. Limpieza de tipos / Data type cleaning

In [0]:
# SPA: Convertir TotalCharges en numérica con los 11 vacíos en 0
# ENG: Convert TotalCharges to numeric with the 11 empty values to 0

vacios = (pdf["TotalCharges"].str.strip() == "").sum()
assert vacios == 11, f"Se esperaban 11 valores vacíos, hay {vacios}"

pdf["TotalCharges"] = pd.to_numeric(pdf["TotalCharges"].str.strip().replace("", "0"))

In [0]:
# SPA: Convertir Churn en 0 y 1
# ENG: Convert Churn to 0 and 1

pdf["Churn"] = pdf["Churn"].map({"Yes": 1, "No": 0})

print(pdf["Churn"].value_counts())

In [0]:
# SPA: Todas las columnas a minúsculas con magia negra (expresiones regulares)
# ENG: All columns to lowercase with dark magic (regex)

def to_snake_case(nombre: str) -> str:
    s = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", nombre)
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)
    return s.lower()

In [0]:
pdf.columns = [to_snake_case(c) for c in pdf.columns]
pdf.columns

### SILVER TABLE CREATION:

In [0]:
# Validar / Validate

def validar_silver(pdf: pd.DataFrame) -> None:
    sin_normalizar = [c for c in pdf.columns if c != c.lower()]
    nulos = pdf.isna().sum().sum()

    assert pdf.shape == (7043, 21), f"Forma inesperada: {pdf.shape}"
    assert not sin_normalizar, f"Columnas sin normalizar: {sin_normalizar}"
    assert pdf["total_charges"].dtype == "float64", f"total_charges es {pdf['total_charges'].dtype}"
    assert pdf["churn"].isin([0, 1]).all(), "churn contiene valores fuera de 0 y 1"
    assert nulos == 0, f"Hay {nulos} nulos en la tabla"


    print(f"OK — {pdf.shape[0]:,} filas x {pdf.shape[1]} columnas")


validar_silver(pdf)

In [0]:
spark.createDataFrame(pdf).write.mode("overwrite").saveAsTable(SILVER_TABLE)

In [0]:
# Validar tabla delta / validate delta table

sdf = spark.table(SILVER_TABLE)

n_filas = sdf.count()
n_cols = len(sdf.columns)

assert n_filas == 7043, f"Filas inesperadas en silver: {n_filas}"
assert n_cols == 21, f"Columnas inesperadas en silver: {n_cols}"

print(f"OK — {n_filas:,} filas x {n_cols} columnas")
sdf.printSchema()

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}"))